In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

In [2]:
dataset = pd.read_csv('../data/cleaned_data.csv')
X = dataset.iloc[:,:-1].values
y = dataset.iloc[:,-1].values

In [3]:
x_train,x_test,y_train,y_test = train_test_split(X,y,train_size=0.8,random_state=0)
y_test_average = np.mean(y_test)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(11377, 211)
(2845, 211)
(11377,)
(2845,)


Modeling Linear regression Model


In [4]:
linear_regressor = LinearRegression()
linear_regressor.fit(x_train,y_train)
y_pred_linear = linear_regressor.predict(x_test)
from sklearn.preprocessing import PolynomialFeatures

Validating Linear Model

In [5]:
r2_linear = r2_score(y_test,y_pred_linear)
r2_linear_train = r2_score(y_train,linear_regressor.predict(x_train))
rmse_linear = root_mean_squared_error(y_test,y_pred_linear)
print(r2_linear,rmse_linear,r2_linear_train)
print((rmse_linear/y_test_average)*100)
mae_linear = mean_absolute_error(y_test, y_pred_linear)
print(mae_linear)

0.6697358833650238 6.572379752600676 0.6953730533583716
70.92944539924483
3.428618871375856


Decision Tree Regression 

In [6]:
tree_regressor = DecisionTreeRegressor(random_state=0,max_depth=10)
tree_regressor.fit(x_train,y_train)
y_pred_tree = tree_regressor.predict(x_test)

Validating Decision Tree Model


In [7]:
r2_tree = r2_score(y_test,y_pred_tree)
r2_tree_train = r2_score(y_train,tree_regressor.predict(x_train))
rmse_tree = root_mean_squared_error(y_test,y_pred_tree)
print(r2_tree,rmse_tree,r2_tree_train)
print((rmse_tree/y_test_average)*100)
mae_tree = mean_absolute_error(y_test, y_pred_tree)
print(mae_tree)

0.6131713872794469 7.1129744583314025 0.7775966789250595
76.76357004003049
3.404781781078519


Random Forest Regression

In [8]:
rf_regressor = RandomForestRegressor(random_state= 0)
rf_regressor.fit(x_train, y_train)
y_rf = rf_regressor.predict(x_test)

Validating Random Forrest Regressor

In [ ]:
r2_rf = r2_score(y_test,y_rf)
r2_rf_train = r2_score(y_train,rf_regressor.predict(x_train))
rmse_rf = root_mean_squared_error(y_test,y_rf)
print(r2_rf,rmse_rf,r2_rf_train)
print((rmse_rf/y_test_average)*100)
mae_rf = mean_absolute_error(y_test, y_rf)
print(mae_rf)

0.961050850751336 5.89898355953791 0.961050850751336
63.66212057842861
2.592025859904594


Standard Scaling 


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_fitValue = scaler.fit(x_train)
X_train_scaled = scaler.transform(x_train)
X_test_scaled = scaler.transform(x_test)

In [11]:
regressor_svr = SVR(kernel = 'rbf')
regressor_svr.fit(X_train_scaled, y_train)
y_pred_svr= regressor_svr.predict(X_test_scaled)

In [ ]:
r2_svr = r2_score(y_test,y_pred_svr)
r2_svr_train = r2_score(y_train,regressor_svr.predict(X_train_scaled))
rmse_svr = root_mean_squared_error(y_test,y_pr(rmse_svr/y_test_average)*100)
mae_svr = mean_absolute_error(y_test, y_pred_svr)
print(mae_svr)

0.5432100657861195 7.729478913216154 0.5577927513131511
83.41691642553658
3.156245974175523


In [13]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_random = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=0),
    param_distributions=param_grid,
    n_iter=20,
    cv=3,
    scoring='r2',
    random_state=0,
    n_jobs=-1
)

rf_random.fit(x_train, y_train)
print(rf_random.best_params_)
print(rf_random.best_score_)

{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': None}
0.7453934248530433


In [14]:
best_rf = rf_random.best_estimator_
y_pred_tuned = best_rf.predict(x_test)

r2_tuned = r2_score(y_test, y_pred_tuned)
rmse_tuned = root_mean_squared_error(y_test, y_pred_tuned)
r2_tuned_train = r2_score(y_train, best_rf.predict(x_train))

print(r2_tuned, rmse_tuned, r2_tuned_train)


0.7374052446506785 5.860504153180795 0.9270691622481269


In [15]:
import pickle
import json

# 1. Save your best estimator from the RandomizedSearchCV grid
with open('../model/model.pkl', 'wb') as f:
    pickle.dump(best_rf, f)

# 2. Extract and save the exact 211 training column sequence (excluding target 'Price')
# Since you did dataset.iloc[:, :-1], your feature columns match dataset.columns[:-1]
feature_columns = list(dataset.columns)[:-1]
with open('../model/columns.json', 'w') as f:
    json.dump(feature_columns, f)

print("Assets successfully serialized inside the model/ directory!")

Assets successfully serialized inside the model/ directory!
